# Batched scans

`opencr.scan` solves one point after another on the reference path. For quasi-static solves of Maxwellian plasmas, `batched_scan` evaluates the rate coefficients and the quasi-static reduction of every $(T_e, n_e)$ pair at once as dense arrays on cunumpy's `xp`: NumPy, or CuPy on a GPU after `cunumpy.set_backend("cupy")`. See [Backends](/OpenCR/guides/backends/) for what is batched and where the execution boundaries are.

In [ ]:
import time

import numpy as np
import plasma_plots

import opencr
from opencr.backends import device
from opencr.backends.batched import batched_scan

model = opencr.load_model("hydrogen/atomic/basic", n_max=16)
temperatures = np.geomspace(0.5, 100.0, 40)
densities = np.geomspace(1e16, 1e22, 7)

In [ ]:
start = time.perf_counter()
data = batched_scan(model, opencr.Conditions(1.0, 1e18), temperatures, densities)
print(
    f"{temperatures.size * densities.size} points in {time.perf_counter() - start:.3f} s on the {device()}"
)
data

The effective ionization coefficient $S_\mathrm{CR}$ against $T_e$ for every density:

In [ ]:
s_cr = data["effective_rate"].sel(target="H+", source="H(n=1)")
with plasma_plots.figure(1, 1) as fig:
    for density in densities:
        s_cr.sel(electron_density=density).plasma.plot.lineout(
            ax=fig[0], title="effective ionization coefficient"
        )
        fig[0].lines[-1].set_label(f"n_e = {density:.0e} m$^{{-3}}$")
    fig[0].set_xscale("log")
    fig[0].set_yscale("log")
    fig[0].set_ylabel(r"$S_\mathrm{CR}$ (m$^3$/s)")
    fig[0].legend(fontsize="small")

The batched result is the same as the per-point reference solve:

In [ ]:
reference = opencr.scan(
    model,
    opencr.Conditions(1.0, 1e18, densities={"H(n=1)": 1e19, "H+": 1e18}),
    lines=False,
    electron_temperature=temperatures[::8],
    electron_density=densities[::3],
)
batched = data["effective_rate"].sel(
    electron_temperature=temperatures[::8], electron_density=densities[::3]
)
print(
    f"largest relative difference: {float(abs(batched / reference['effective_rate'] - 1).max()):.1e}"
)